# Grade Headnotes

In this notebook, we will run the Llama3 model locally using Ollama to grade case headnotes written by law experts. Specifically, we will provide the model with a thorough rubric and a few examples. Then, we will then feed to it the entire dataset of 7k headnotes and full judgements, and have it update our dataset with the grades. We will also have the model generate explanations for why they chose that grade.

## Import Libraries

In [1]:
import ollama
import pandas as pd

## Load Dataset

In [2]:
df = pd.read_json("../data/combined_original_data.json").T
df.head(5)

,judgement,summary
1,Appeal No. LXVI of 1949.\nAppeal from the High...,The charge created in respect of municipal pro...
2,XXIX of 1950.\nApplication under article 32 of...,Section 7 (1) (c) of the East Punjab Public Sa...
3,XXXVII of 1950.\nApplication under article 32 ...,"Section 4, sub section\n(1) (c), of the East P..."
4,No. XVI of 1950.\nAppli cation under article 3...,"Held, by the Full Court (i) (overruling a prel..."
5,Civil Appeal No. 8 of 1951.\nAppeal from the j...,S and B were sons of two brothers respectively...


In [3]:
df = df.dropna()
df = df.drop_duplicates(subset=['summary'])
df['summary'] = df['summary'].str.strip()
df['judgement'] = df['judgement'].str.strip()
df.describe()

,judgement,summary
count,7100,7100
unique,7100,7100
top,Appeal No. LXVI of 1949.\nAppeal from the High...,The charge created in respect of municipal pro...
freq,1,1


## Set up the prompt

In [4]:
background_prompt = """
Background:
Case judgments represent rulings issued by judges in judicial proceedings, often comprising extensive transcripts spanning multiple pages. 
Headnotes present a judgment in an organized manner to aid in determining the most relevant facts, recognizing the legal issues involved, 
the parties arguments and contentions, providing a concise discussion of the judgment, and outlining the reasoning behind the decision. 
An excellent headnote captures the core of the judgment properly, succinctly, and completely.Headnotes are vital for legal researchers, 
practicing advocates, and the general public in reaching a decision.

Task:
You are given several thousand Indian Supreme Court judgments and headnotes written by legal experts. Your job is to classify each summary 
as great, medium, or poor based on a rubric I will give you. For each classification, you also need to provide reasoning for why you chose that. 

Rubric: 
- Word count: The headnote should be between 300 and 800 words. 
- Style: The headnote should be written in the past tense. It should have limited usage of proper nouns as well as irrelevant dates and statistics.
- Metadata: The headnote should include the case name, judgement number, the court where the judgement was given, the judge(s) hearing the case, 
and the date of the judgement. 
- Introduction: The headnote should begin with the overall field of law that the case deals with.
- Key facts: The headnote should include facts surrounding the nature of the litigation, who filed the case, why the case was filed, 
what remedy the filer is asking for, and what has already been decided until now. 
- Key arguments: The headnote should include both sides' submissions and reasons to support their position. Only relevant arguments to the 
issue and conclusion should be included.
- Relevant laws: The headnote should include references to relevant legislation and concepts to support the case judgment/ruling. 
If applicable, it should also include topics or questions that judges had set aside for discussion and decision-making.
- Relevant cases: If applicable, the headnote can contain references to other cases that were considered by the judge(s), and whether those 
were followed or otherwise. 
- Conclusion: The headnote should conclude with the case's conclusion and procedural disposition (ex: permitted, dismissed, reversed, 
remanded, affirmed, etc.)
"""

## Call the model

In [5]:
! ollama pull llama3

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB                         
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB                         
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B                         
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B                         
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [6]:
def classify_headnote(judgement, headnote):
    prompt = f'{background_prompt}\n\nJudgement: {judgement}\n\nHeadnote: {headnote}\n\nGrade the headnote as good, medium, or poor, and provide your reasoning:'
    response = ollama.chat(model="llama3", messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"]

In [7]:
final_df_list = []
for index, row in df.iterrows():
    judgement = row['judgement']
    headnote = row['summary']
    response = classify_headnote(judgement, headnote)
    final_df_list.append({"judgement": judgement, "headnote": headnote, "response": response})
    if len(final_df_list) == 5:
        break

In [8]:
df_final = pd.DataFrame(final_df_list)
df_final.head(5)

,judgement,headnote,response
0,Appeal No. LXVI of 1949.\nAppeal from the High...,The charge created in respect of municipal pro...,I would grade the headnote as GOOD.\n\nHere's ...
1,XXIX of 1950.\nApplication under article 32 of...,Section 7 (1) (c) of the East Punjab Public Sa...,"I would grade the headnote as ""good"". Here's w..."
2,XXXVII of 1950.\nApplication under article 32 ...,"Section 4, sub section\n(1) (c), of the East P...","I would grade the headnote as ""good"". Here's w..."
3,No. XVI of 1950.\nAppli cation under article 3...,"Held, by the Full Court (i) (overruling a prel...","I would grade the headnote as ""good"".\n\nHere'..."
4,Civil Appeal No. 8 of 1951.\nAppeal from the j...,S and B were sons of two brothers respectively...,"I would grade the headnote as ""poor"".\n\nHere'..."


In [10]:
df_final.to_json("../data/llama_classification_outputs.json", orient="index")